# ARC_ATLAS v4 ATLAS-Heavy Replay Resume v2

This notebook replaces the prior pure ATLAS-only fine-tune.

Purpose:

- start from the best tri-planar `goal08` checkpoints
- fine-tune with **ATLAS-heavy mixed replay** instead of pure ATLAS-only training
- freeze only the earliest encoder blocks while keeping BatchNorm frozen
- evaluate on the **same mixed validation split** plus its ATLAS subset so the comparisons stay clean
- loosen the fine-tune enough that the branches can still move


In [1]:
from pathlib import Path
import sys
import time
import json

candidates = [
    Path.cwd(),
    Path.cwd() / "ARC_ATLAS_Combined" / "ARC_ATLAS_Train_v4",
    Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4"),
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src" / "training_v2_slice_blocks.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate ARC_ATLAS_Train_v4/src/training_v2_slice_blocks.py")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import training_v2_slice_blocks as seg

TRAIN_DIR = PROJECT_ROOT / "data" / "splits" / "90_10_random" / "train"
FULL_MANIFEST = TRAIN_DIR / "manifest.csv"
SOURCE_RUN = PROJECT_ROOT / "runs" / "20260422_171449_slice_blocks_triplanar_goal08"
RUN_DIR = PROJECT_ROOT / "runs" / f"{time.strftime('%Y%m%d_%H%M%S')}_slice_blocks_resume_atlas_replay_looserfreeze"
RUN_DIR.mkdir(parents=True, exist_ok=True)

ATLAS_SLUG = "ATLAS-Images-f0d7431e"
ARC_SLUG = "ARC-combined-t1-raw-ab0d1794"
APPROX_SLUG = "Approx-Numeracy-Processed"
SOURCE_WEIGHT_MAP = {
    ATLAS_SLUG: 3.5,
    ARC_SLUG: 1.5,
    APPROX_SLUG: 1.0,
}

AXIS_NAMES = {0: "sagittal", 1: "coronal", 2: "axial"}
AXES_TO_TRAIN = (2, 0, 1)

COMMON_CFG = dict(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_DIR / "t1",
    MASKS_DIR=TRAIN_DIR / "masks",
    MANIFEST_PATH=FULL_MANIFEST,
    TARGET_SHAPE=(192, 224, 192),
    RESAMPLE_TO_TARGET=False,
    VALIDATION_SPLIT=0.15,
    BLOCK_DEPTH=7,
    SLICE_STRIDE=1,
    TOTAL_EPOCHS=18,
    INITIAL_LR=6e-6,
    MIN_LR=1e-6,
    WEIGHT_DECAY=5e-6,
    MAX_GRAD_NORM=0.75,
    MIXED_PRECISION=False,
    JIT_COMPILE=False,
    BASE_FILTERS=12,
    UNET_DEPTH=4,
    DROPOUT_RATE=0.06,
    L2_REG=1e-4,
    BALANCED_CASE_SAMPLING=True,
    SOURCE_BALANCED_SAMPLING=True,
    SOURCE_SAMPLING_WEIGHTS=tuple(SOURCE_WEIGHT_MAP.items()),
    SIZE_AWARE_SAMPLING=True,
    SIZE_BUCKET_EDGES=(100, 1000, 10000),
    SIZE_BUCKET_PROBS=(0.42, 0.28, 0.18, 0.12),
    STEPS_PER_EPOCH=420,
    VALIDATION_STEPS=18,
    POSITIVE_WEIGHT=28.0,
    BCE_WEIGHT=0.38,
    DICE_WEIGHT=0.46,
    FOCAL_TVERSKY_WEIGHT=0.16,
    TVERSKY_ALPHA=0.58,
    TVERSKY_BETA=0.42,
    FOCAL_TVERSKY_GAMMA=1.33,
    LESION_SLICE_WEIGHT=1.35,
    EMPTY_SLICE_WEIGHT=0.95,
    DICE_ON_LESION_SLICES_ONLY=False,
    POSITIVE_TOPK_WEIGHT=0.04,
    POSITIVE_TOPK_FRACTION=0.20,
    SMALL_LESION_BOOST_REFERENCE=10000.0,
    SMALL_LESION_BOOST_MAX=1.90,
    AUGMENT=True,
    AUG_FLIP_PROB=0.5,
    AUG_INTENSITY_SCALE=0.04,
    AUG_INTENSITY_SHIFT=0.02,
    AUG_NOISE_STD=0.004,
    DECISION_THRESHOLD=0.50,
    VAL_THRESHOLD_SWEEP=(0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.80),
    WHOLE_BRAIN_VAL_EVERY_N_EPOCHS=1,
    WHOLE_BRAIN_VAL_MAX_CASES=None,
    SAVE_VAL_PREDICTIONS=True,
    NUM_VAL_PREDICTIONS=4,
    EARLY_STOPPING_PATIENCE=6,
    EARLY_STOPPING_MIN_DELTA=0.0003,
    RESTORE_BEST_WEIGHTS=True,
    TARGET_WHOLE_DICE=None,
    FROZEN_LAYER_PREFIXES=("enc1_", "enc2_"),
    FREEZE_BATCHNORM=True,
    FIT_VERBOSE=2,
)

def make_axis_cfg(axis: int):
    name = AXIS_NAMES[axis]
    axis_dir = RUN_DIR / f"axis{axis}_{name}"
    initial_weights = SOURCE_RUN / f"axis{axis}_{name}" / "callbacks" / "best_slice_block.weights.h5"
    if not initial_weights.exists():
        raise FileNotFoundError(initial_weights)
    return seg.SliceBlockTrainingConfig(
        **COMMON_CFG,
        SLICE_AXIS=axis,
        INITIAL_WEIGHTS_PATH=initial_weights,
        MODEL_DIR=axis_dir / "models",
        CALLBACKS_DIR=axis_dir / "callbacks",
    )

axis_cfgs = {axis: make_axis_cfg(axis) for axis in AXES_TO_TRAIN}

print(f"Project root: {PROJECT_ROOT}")
print(f"Source tri-planar run: {SOURCE_RUN}")
print(f"Resume run dir: {RUN_DIR}")
print(f"Training manifest: {FULL_MANIFEST}")
print(f"Source replay weights: {SOURCE_WEIGHT_MAP}")
for axis, cfg in axis_cfgs.items():
    print(f"axis {axis} ({AXIS_NAMES[axis]}): init={cfg.INITIAL_WEIGHTS_PATH}")
    print(f"  frozen prefixes: {cfg.FROZEN_LAYER_PREFIXES} | freeze_batchnorm={cfg.FREEZE_BATCHNORM}")
    print(f"  input per brain: (num_slices, {cfg.input_shape[0]}, {cfg.input_shape[1]}, {cfg.input_shape[2]})")
    print(f"  checkpoints: {cfg.checkpoint_path}")


2026-04-30 14:14:19.769789: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Project root: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4
Source tri-planar run: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260422_171449_slice_blocks_triplanar_goal08
Resume run dir: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260430_141421_slice_blocks_resume_atlas_replay_looserfreeze
Training manifest: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv
Source replay weights: {'ATLAS-Images-f0d7431e': 3.5, 'ARC-combined-t1-raw-ab0d1794': 1.5, 'Approx-Numeracy-Processed': 1.0}
axis 2 (axial): init=/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260422_171449_slice_blocks_triplanar_goal08/axis2_axial/callbacks/best_slice_block.weights.h5
  frozen prefixes: ('enc1_', 'enc2_') | freeze_batchnorm=True
  input per brain: (num_slices, 192, 224, 7)
  checkpoints: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLA

In [2]:
# Verify checkpoints and inspect the mixed split that will be reused for training + evaluation.
import pandas as pd

reference_cfg = axis_cfgs[2 if 2 in axis_cfgs else AXES_TO_TRAIN[0]]

for axis, cfg in axis_cfgs.items():
    model = seg.build_slice_block_model(cfg)
    model.load_weights(str(cfg.INITIAL_WEIGHTS_PATH))
    seg.apply_trainable_policy(model, cfg)
    trainable_params = int(sum(seg.np.prod(w.shape) for w in model.trainable_weights))
    frozen_params = int(sum(seg.np.prod(w.shape) for w in model.non_trainable_weights))
    print(f"Loaded axis {axis} checkpoint: {cfg.INITIAL_WEIGHTS_PATH}")
    print(f"  total params: {model.count_params():,}")
    print(f"  trainable params after freeze policy: {trainable_params:,}")
    print(f"  non-trainable params after freeze policy: {frozen_params:,}")
    del model
    seg.tf.keras.backend.clear_session()

mixed_cases = seg.load_cases(reference_cfg)
mixed_train_cases, mixed_val_cases, mixed_lesion_sizes = seg.split_cases(mixed_cases, reference_cfg)
size_by_id = {case.case_id: size for case, size in zip(mixed_cases, mixed_lesion_sizes)}
mixed_val_atlas_cases = [case for case in mixed_val_cases if case.source == ATLAS_SLUG]
mixed_val_arc_cases = [case for case in mixed_val_cases if case.source == ARC_SLUG]
mixed_val_approx_cases = [case for case in mixed_val_cases if case.source == APPROX_SLUG]

def split_frame(split_name, split_cases):
    return pd.DataFrame([
        {
            "split": split_name,
            "source": case.source,
            "case_id": case.case_id,
            "lesion_voxels": int(size_by_id[case.case_id]),
            "lesion_group": seg.lesion_size_group(int(size_by_id[case.case_id])),
        }
        for case in split_cases
    ])

split_df = pd.concat([
    split_frame("train", mixed_train_cases),
    split_frame("val", mixed_val_cases),
], ignore_index=True)

print(f"Mixed cases: {len(mixed_cases)}")
print(f"Mixed train/val: {len(mixed_train_cases)} / {len(mixed_val_cases)}")
print(f"Mixed val by source: ATLAS={len(mixed_val_atlas_cases)} ARC={len(mixed_val_arc_cases)} Approx={len(mixed_val_approx_cases)}")
display(split_df.groupby(["split", "source", "lesion_group"]).size().unstack(fill_value=0))


I0000 00:00:1777580061.850298  142886 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1777580061.851666  142886 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22122 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2026-04-30 14:14:22,756 - SliceBlockTrainer - INFO - Applied fine-tune freezing: prefixes=('enc1_', 'enc2_') freeze_batchnorm=True frozen_layers=28/77


Loaded axis 2 checkpoint: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260422_171449_slice_blocks_triplanar_goal08/axis2_axial/callbacks/best_slice_block.weights.h5
  total params: 1,108,081
  trainable params after freeze policy: 1,093,837
  non-trainable params after freeze policy: 14,244


2026-04-30 14:14:23,121 - SliceBlockTrainer - INFO - Applied fine-tune freezing: prefixes=('enc1_', 'enc2_') freeze_batchnorm=True frozen_layers=28/77


Loaded axis 0 checkpoint: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260422_171449_slice_blocks_triplanar_goal08/axis0_sagittal/callbacks/best_slice_block.weights.h5
  total params: 1,108,081
  trainable params after freeze policy: 1,093,837
  non-trainable params after freeze policy: 14,244


2026-04-30 14:14:23,483 - SliceBlockTrainer - INFO - Applied fine-tune freezing: prefixes=('enc1_', 'enc2_') freeze_batchnorm=True frozen_layers=28/77
2026-04-30 14:14:23,599 - SliceBlockTrainer - INFO - Using manifest: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv
2026-04-30 14:14:23,618 - SliceBlockTrainer - INFO - Loaded 866 cases from manifest


Loaded axis 1 checkpoint: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260422_171449_slice_blocks_triplanar_goal08/axis1_coronal/callbacks/best_slice_block.weights.h5
  total params: 1,108,081
  trainable params after freeze policy: 1,093,837
  non-trainable params after freeze policy: 14,244


2026-04-30 14:16:38,401 - SliceBlockTrainer - INFO - Split cases: train=737 val=129


Mixed cases: 866
Mixed train/val: 737 / 129
Mixed val by source: ATLAS=88 ARC=28 Approx=13


lesion_group                        10000_plus  1000_9999  100_999  1_99  none
split source                                                                  
train ARC-combined-t1-raw-ab0d1794         144         13        4     0     1
      ATLAS-Images-f0d7431e                186        167      126    14     1
      Approx-Numeracy-Processed             36         19       20     5     1
val   ARC-combined-t1-raw-ab0d1794          25          2        1     0     0
      ATLAS-Images-f0d7431e                 33         30       22     3     0
      Approx-Numeracy-Processed              6          3        3     1     0

In [3]:
# Sanity check one ATLAS validation case under the replay fine-tune config.
sanity_case = mixed_val_atlas_cases[0] if mixed_val_atlas_cases else mixed_val_cases[0]
reference_cfg = axis_cfgs[2 if 2 in axis_cfgs else AXES_TO_TRAIN[0]]
image, mask, _ = seg.load_case_arrays(sanity_case, reference_cfg)
x, y = seg.make_slice_blocks(image, mask, reference_cfg)
model = seg.build_slice_block_model(reference_cfg)
seg.apply_trainable_policy(model, reference_cfg)

print(f"Sanity case: {sanity_case.case_id}")
print(f"Prepared brain: image={image.shape}, mask={mask.shape}")
print(f"One-brain batch: x={x.shape}, y={y.shape}")
print(f"Lesion voxels in sanity case: {int(y.sum())}")
print(f"Model params: {model.count_params():,}")
print(f"Trainable tensors after freeze policy: {len(model.trainable_weights)}")

model.load_weights(str(reference_cfg.INITIAL_WEIGHTS_PATH))
pred = model.predict(x[:4], verbose=0)
print(f"Loaded checkpoint prediction range: min={float(pred.min()):.6f}, max={float(pred.max()):.6f}")

del model
seg.tf.keras.backend.clear_session()


2026-04-30 14:16:39,085 - SliceBlockTrainer - INFO - Applied fine-tune freezing: prefixes=('enc1_', 'enc2_') freeze_batchnorm=True frozen_layers=28/77


Sanity case: ATLAS-Images-f0d7431e__ATLAS-Images-f0d7431e__sub-r011s017_ses-1_space-MNI152NLin2009aSym_T1w_MNI_norm
Prepared brain: image=(192, 224, 192), mask=(192, 224, 192)
One-brain batch: x=(192, 192, 224, 7), y=(192, 192, 224, 1)
Lesion voxels in sanity case: 823
Model params: 1,108,081
Trainable tensors after freeze policy: 16


2026-04-30 14:16:39.449048: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d8ff80481c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-30 14:16:39.449075: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-04-30 14:16:39.449082: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (1): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-04-30 14:16:39.476331: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-04-30 14:16:39.604007: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91001


Loaded checkpoint prediction range: min=0.000210, max=0.001093


I0000 00:00:1777580201.153297  143191 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [4]:
# Launch ATLAS-heavy replay fine-tune for each axis.
# This keeps replay, but loosens the freeze policy so enc3 and deeper layers can adapt.
histories = {}
for axis, cfg in axis_cfgs.items():
    print("=" * 100)
    print(f"ATLAS-heavy replay fine-tune v2: axis {axis} ({AXIS_NAMES[axis]})")
    print(f"Callbacks dir: {cfg.CALLBACKS_DIR}")
    histories[axis] = seg.train_slice_block_model(cfg)
    seg.tf.keras.backend.clear_session()
    print(f"Finished axis {axis}. Best checkpoint: {cfg.checkpoint_path}")


2026-04-30 14:16:41,305 - SliceBlockTrainer - WARNING - Could not set memory growth on PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'): Physical devices cannot be modified after being initialized
2026-04-30 14:16:41,306 - SliceBlockTrainer - WARNING - Could not set memory growth on PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'): Physical devices cannot be modified after being initialized
2026-04-30 14:16:41,306 - SliceBlockTrainer - INFO - Using manifest: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv
2026-04-30 14:16:41,324 - SliceBlockTrainer - INFO - Loaded 866 cases from manifest


ATLAS-heavy replay fine-tune v2: axis 2 (axial)
Callbacks dir: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260430_141421_slice_blocks_resume_atlas_replay_looserfreeze/axis2_axial/callbacks


2026-04-30 14:18:37,943 - SliceBlockTrainer - INFO - Split cases: train=737 val=129
2026-04-30 14:18:37,958 - SliceBlockTrainer - INFO - Balanced case sampling enabled: source_balanced=True source_weights={'ATLAS-Images-f0d7431e': 3.5, 'ARC-combined-t1-raw-ab0d1794': 1.5, 'Approx-Numeracy-Processed': 1.0} size_aware=True size_edges=(100, 1000, 10000) size_probs=(0.42, 0.28, 0.18, 0.12)
2026-04-30 14:18:38,152 - SliceBlockTrainer - INFO - Loading initial weights from /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260422_171449_slice_blocks_triplanar_goal08/axis2_axial/callbacks/best_slice_block.weights.h5
2026-04-30 14:18:38,251 - SliceBlockTrainer - INFO - Copied initial weights to /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260430_141421_slice_blocks_resume_atlas_replay_looserfreeze/axis2_axial/callbacks/baseline_initial.weights.h5
2026-04-30 14:18:38,252 - SliceBlockTrainer - INFO - Applied fine-tune freezing: prefixes=('enc1

Epoch 1/18


E0000 00:00:1777580320.729455  142886 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/SliceBlock2p5D_UNet_1/enc1_dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-04-30 14:32:05,752 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 16/129
2026-04-30 14:32:35,822 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 32/129
2026-04-30 14:33:05,850 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 48/129
2026-04-30 14:33:35,966 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 64/129
2026-04-30 14:34:05,947 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 80/129
2026-04-30 14:34:35,912 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 96/129
2026-04-30 14:35:05,991 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 112/129
2026-04-30 14:35:36,101

...


Epoch 14: val_whole_dice_hard_best_thr_score did not improve from 0.37012
420/420 - 965s - 2s/step - dice_coefficient: 0.0566 - foreground_fraction: 0.0059 - hard_dice_metric: 0.6224 - loss: 0.6154 - val_dice_coefficient: 0.0812 - val_foreground_fraction: 0.0069 - val_hard_dice_metric: 0.6298 - val_loss: 0.6589 - val_whole_dice_soft_macro: 0.2791 - val_whole_dice_hard: 0.3225 - val_whole_dice_hard_best_thr: 0.8000 - val_whole_dice_hard_best_thr_score: 0.3618
Epoch 15/18


2026-05-01 01:38:30,794 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 16/129
2026-05-01 01:38:55,855 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 32/129
2026-05-01 01:39:21,024 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 48/129
2026-05-01 01:39:46,038 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 64/129
2026-05-01 01:40:11,087 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 80/129
2026-05-01 01:40:36,011 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 96/129
2026-05-01 01:41:01,106 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 112/129
2026-05-01 01:41:26,220 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 128/129
2026-05-01 01:41:27,796 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 129/129
2026-05-01 01:41:27,800 - SliceBlockTrainer - INFO - Whole-brain slice-block val @epoch 14: soft=0.27926 hard@0.50=


Epoch 15: val_whole_dice_hard_best_thr_score did not improve from 0.37012
420/420 - 965s - 2s/step - dice_coefficient: 0.0634 - foreground_fraction: 0.0066 - hard_dice_metric: 0.6460 - loss: 0.6188 - val_dice_coefficient: 0.0823 - val_foreground_fraction: 0.0071 - val_hard_dice_metric: 0.6320 - val_loss: 0.6567 - val_whole_dice_soft_macro: 0.2793 - val_whole_dice_hard: 0.3224 - val_whole_dice_hard_best_thr: 0.8000 - val_whole_dice_hard_best_thr_score: 0.3613
Epoch 16/18


2026-05-01 01:54:37,123 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 16/129
2026-05-01 01:55:02,221 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 32/129
2026-05-01 01:55:27,530 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 48/129
2026-05-01 01:55:52,623 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 64/129
2026-05-01 01:56:17,742 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 80/129
2026-05-01 01:56:43,037 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 96/129
2026-05-01 01:57:08,280 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 112/129
2026-05-01 01:57:33,380 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 128/129
2026-05-01 01:57:34,950 - SliceBlockTrainer - INFO - Whole-brain slice-block val progress: 129/129
2026-05-01 01:57:34,954 - SliceBlockTrainer - INFO - Whole-brain slice-block val @epoch 15: soft=0.27810 hard@0.50=


Epoch 16: val_whole_dice_hard_best_thr_score did not improve from 0.37012
420/420 - 967s - 2s/step - dice_coefficient: 0.0624 - foreground_fraction: 0.0061 - hard_dice_metric: 0.6628 - loss: 0.6185 - val_dice_coefficient: 0.0817 - val_foreground_fraction: 0.0069 - val_hard_dice_metric: 0.6267 - val_loss: 0.6586 - val_whole_dice_soft_macro: 0.2781 - val_whole_dice_hard: 0.3233 - val_whole_dice_hard_best_thr: 0.8000 - val_whole_dice_hard_best_thr_score: 0.3619
Epoch 16: early stopping
Restoring model weights from the end of the best epoch: 10.
Finished axis 1. Best checkpoint: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260430_141421_slice_blocks_resume_atlas_replay_looserfreeze/axis1_coronal/callbacks/best_slice_block.weights.h5


In [5]:
# Review the per-axis fine-tune logs.
import pandas as pd

for axis, cfg in axis_cfgs.items():
    print("=" * 100)
    print(f"axis {axis} ({AXIS_NAMES[axis]})")
    train_log = cfg.CALLBACKS_DIR / "training_log.csv"
    whole_val_log = cfg.CALLBACKS_DIR / "whole_val_summary.jsonl"
    if train_log.exists():
        display(pd.read_csv(train_log).tail(8))
    else:
        print(f"No training log yet: {train_log}")
    if whole_val_log.exists():
        rows = [json.loads(line) for line in whole_val_log.read_text().splitlines() if line.strip()]
        display(pd.DataFrame(rows).tail(8))
    else:
        print(f"No whole-brain validation log yet: {whole_val_log}")


axis 2 (axial)


,epoch,dice_coefficient,foreground_fraction,hard_dice_metric,loss,val_dice_coefficient,val_foreground_fraction,val_hard_dice_metric,val_loss,val_whole_dice_hard,val_whole_dice_hard_best_thr,val_whole_dice_hard_best_thr_score,val_whole_dice_soft_macro
7,7,0.054197,0.008414,0.557751,0.641001,0.085699,0.008212,0.561632,0.659318,0.269375,0.8,0.312650,0.230496
8,8,0.050005,0.008289,0.541200,0.650162,0.084385,0.007930,0.565251,0.657844,0.272626,0.8,0.317812,0.227163
9,9,0.056126,0.008325,0.546668,0.641765,0.082890,0.008641,0.549658,0.654107,0.261656,0.8,0.306976,0.218684
10,10,0.042288,0.007663,0.549497,0.646770,0.083861,0.008320,0.550764,0.655266,0.265863,0.8,0.311313,0.222243
11,11,0.048045,0.008025,0.556634,0.635584,0.084879,0.008441,0.562420,0.655647,0.266140,0.8,0.310874,0.225977
12,12,0.043298,0.008635,0.543218,0.641935,0.083129,0.008910,0.523289,0.654508,0.255673,0.8,0.301475,0.216947
13,13,0.048979,0.007833,0.536163,0.630775,0.084415,0.008544,0.548378,0.655909,0.262724,0.8,0.308237,0.223327
14,14,0.056717,0.008486,0.560300,0.632438,0.084916,0.008491,0.555746,0.655951,0.264507,0.8,0.310388,0.224729


,epoch,elapsed_sec,n_cases,val_whole_dice_soft_macro,val_whole_dice_hard,val_whole_dice_hard_best_thr,val_whole_dice_hard_best_thr_score,pred_max_p90,pred_hard_voxels_median
7,7,236.224189,129,0.230496,0.269375,0.8,0.312650,0.991173,39668.0
8,8,236.704535,129,0.227163,0.272626,0.8,0.317812,0.990756,39538.0
9,9,236.315620,129,0.218684,0.261656,0.8,0.306976,0.990962,44947.0
10,10,235.694467,129,0.222243,0.265863,0.8,0.311313,0.990401,42616.0
11,11,236.186471,129,0.225977,0.266140,0.8,0.310874,0.990281,42352.0
12,12,238.442055,129,0.216947,0.255673,0.8,0.301475,0.989887,47525.0
13,13,242.867774,129,0.223327,0.262724,0.8,0.308237,0.989817,44406.0
14,14,241.060751,129,0.224729,0.264507,0.8,0.310388,0.989738,43362.0


axis 0 (sagittal)


,epoch,dice_coefficient,foreground_fraction,hard_dice_metric,loss,val_dice_coefficient,val_foreground_fraction,val_hard_dice_metric,val_loss,val_whole_dice_hard,val_whole_dice_hard_best_thr,val_whole_dice_hard_best_thr_score,val_whole_dice_soft_macro
4,4,0.031072,0.007032,0.518659,0.665690,0.055416,0.008175,0.494192,0.683311,0.256425,0.80,0.302713,0.208019
5,5,0.035024,0.007203,0.523839,0.653504,0.057690,0.006144,0.597708,0.699136,0.287787,0.70,0.321950,0.237583
6,6,0.031405,0.006242,0.520141,0.660759,0.058384,0.006314,0.570672,0.699815,0.284024,0.70,0.319251,0.238618
7,7,0.036495,0.007978,0.497547,0.661883,0.056535,0.006890,0.484155,0.693981,0.266865,0.80,0.303584,0.222088
8,8,0.033795,0.007681,0.473536,0.671835,0.056849,0.005745,0.534242,0.707223,0.278102,0.65,0.311983,0.230989
9,9,0.037166,0.008860,0.455907,0.665881,0.056221,0.006677,0.518185,0.698737,0.268469,0.80,0.305354,0.225202
10,10,0.028721,0.007138,0.496315,0.662991,0.054609,0.007962,0.441559,0.692258,0.248074,0.80,0.288509,0.205786
11,11,0.033164,0.008512,0.481866,0.653678,0.055273,0.006440,0.572622,0.709648,0.278168,0.70,0.309525,0.234043


,epoch,elapsed_sec,n_cases,val_whole_dice_soft_macro,val_whole_dice_hard,val_whole_dice_hard_best_thr,val_whole_dice_hard_best_thr_score,pred_max_p90,pred_hard_voxels_median
4,4,170.080105,129,0.208019,0.256425,0.80,0.302713,0.999601,42534.0
5,5,175.603923,129,0.237583,0.287787,0.70,0.321950,0.999426,22672.0
6,6,168.963327,129,0.238618,0.284024,0.70,0.319251,0.999456,22784.0
7,7,168.639539,129,0.222088,0.266865,0.80,0.303584,0.999615,30167.0
8,8,168.722354,129,0.230989,0.278102,0.65,0.311983,0.999530,24648.0
9,9,170.152300,129,0.225202,0.268469,0.80,0.305354,0.999474,30610.0
10,10,176.851672,129,0.205786,0.248074,0.80,0.288509,0.999416,45826.0
11,11,178.515188,129,0.234043,0.278168,0.70,0.309525,0.999430,23927.0


axis 1 (coronal)


,epoch,dice_coefficient,foreground_fraction,hard_dice_metric,loss,val_dice_coefficient,val_foreground_fraction,val_hard_dice_metric,val_loss,val_whole_dice_hard,val_whole_dice_hard_best_thr,val_whole_dice_hard_best_thr_score,val_whole_dice_soft_macro
8,8,0.055977,0.006535,0.643365,0.637362,0.081619,0.006794,0.613629,0.654887,0.321447,0.8,0.359497,0.275248
9,9,0.063057,0.006547,0.621287,0.627631,0.081507,0.006631,0.657382,0.656056,0.332477,0.8,0.370119,0.280268
10,10,0.047284,0.005489,0.652135,0.636002,0.082417,0.007328,0.599015,0.642700,0.312656,0.8,0.354479,0.258177
11,11,0.055414,0.006192,0.634851,0.617836,0.082894,0.007330,0.635644,0.652624,0.321186,0.8,0.360000,0.278901
12,12,0.052328,0.006188,0.642616,0.621564,0.081774,0.007059,0.625561,0.656236,0.320279,0.8,0.359869,0.278812
13,13,0.056584,0.005934,0.622427,0.615411,0.081243,0.006891,0.629758,0.658908,0.322484,0.8,0.361766,0.279108
14,14,0.063361,0.006576,0.645951,0.618752,0.082279,0.007105,0.631962,0.656710,0.322357,0.8,0.361279,0.279259
15,15,0.062412,0.006092,0.662750,0.618492,0.081725,0.006873,0.626683,0.658568,0.323336,0.8,0.361945,0.278102


,epoch,elapsed_sec,n_cases,val_whole_dice_soft_macro,val_whole_dice_hard,val_whole_dice_hard_best_thr,val_whole_dice_hard_best_thr_score,pred_max_p90,pred_hard_voxels_median
8,8,204.179630,129,0.275248,0.321447,0.8,0.359497,0.994537,23194.0
9,9,204.780721,129,0.280268,0.332477,0.8,0.370119,0.994468,22537.0
10,10,202.934186,129,0.258177,0.312656,0.8,0.354479,0.994491,27334.0
11,11,202.404045,129,0.278901,0.321186,0.8,0.360000,0.994810,24060.0
12,12,202.683977,129,0.278812,0.320279,0.8,0.359869,0.994679,23979.0
13,13,202.821404,129,0.279108,0.322484,0.8,0.361766,0.994722,23928.0
14,14,202.347828,129,0.279259,0.322357,0.8,0.361279,0.994802,24628.0
15,15,203.419683,129,0.278102,0.323336,0.8,0.361945,0.994831,23560.0


In [6]:
# Evaluate the fine-tuned ensemble on the mixed validation split and its ATLAS subset.
trained_axes = [axis for axis, cfg in axis_cfgs.items() if cfg.checkpoint_path.exists()]
if not trained_axes:
    raise FileNotFoundError("No fine-tuned checkpoints exist yet. Run the training cell first.")

trained_cfgs = [axis_cfgs[axis] for axis in trained_axes]
weights_paths = [axis_cfgs[axis].checkpoint_path for axis in trained_axes]

atlas_subset_summary = seg.evaluate_slice_block_ensemble(
    configs=trained_cfgs,
    weights_paths=weights_paths,
    cases=mixed_val_atlas_cases,
    out_dir=RUN_DIR / "atlas_subset_eval",
    thresholds=COMMON_CFG["VAL_THRESHOLD_SWEEP"],
    decision_threshold=COMMON_CFG["DECISION_THRESHOLD"],
    save_predictions=6,
)

mixed_summary = seg.evaluate_slice_block_ensemble(
    configs=trained_cfgs,
    weights_paths=weights_paths,
    cases=mixed_val_cases,
    out_dir=RUN_DIR / "mixed_ensemble_eval",
    thresholds=COMMON_CFG["VAL_THRESHOLD_SWEEP"],
    decision_threshold=COMMON_CFG["DECISION_THRESHOLD"],
    save_predictions=6,
)

print("ATLAS subset of mixed validation summary")
print(json.dumps(atlas_subset_summary, indent=2))
print()
print("Full mixed-domain validation summary")
print(json.dumps(mixed_summary, indent=2))


2026-05-01 01:59:01,975 - SliceBlockTrainer - INFO - Slice-block ensemble val progress: 16/88
2026-05-01 02:00:08,078 - SliceBlockTrainer - INFO - Slice-block ensemble val progress: 32/88
2026-05-01 02:01:14,359 - SliceBlockTrainer - INFO - Slice-block ensemble val progress: 48/88
2026-05-01 02:02:20,640 - SliceBlockTrainer - INFO - Slice-block ensemble val progress: 64/88
2026-05-01 02:03:26,880 - SliceBlockTrainer - INFO - Slice-block ensemble val progress: 80/88
2026-05-01 02:04:00,065 - SliceBlockTrainer - INFO - Slice-block ensemble val progress: 88/88
2026-05-01 02:04:00,068 - SliceBlockTrainer - INFO - Slice-block ensemble val: soft=0.15133 hard@0.50=0.26685 best_thr_med=0.62 best_thr_score=0.32445
2026-05-01 02:05:16,479 - SliceBlockTrainer - INFO - Slice-block ensemble val progress: 16/129
2026-05-01 02:06:22,775 - SliceBlockTrainer - INFO - Slice-block ensemble val progress: 32/129
2026-05-01 02:07:28,949 - SliceBlockTrainer - INFO - Slice-block ensemble val progress: 48/129


ATLAS subset of mixed validation summary
{
  "elapsed_sec": 383.96516489982605,
  "n_cases": 88,
  "weights_paths": [
    "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260430_141421_slice_blocks_resume_atlas_replay_looserfreeze/axis2_axial/callbacks/best_slice_block.weights.h5",
    "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260430_141421_slice_blocks_resume_atlas_replay_looserfreeze/axis0_sagittal/callbacks/best_slice_block.weights.h5",
    "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260430_141421_slice_blocks_resume_atlas_replay_looserfreeze/axis1_coronal/callbacks/best_slice_block.weights.h5"
  ],
  "axes": [
    2,
    0,
    1
  ],
  "block_depths": [
    7,
    7,
    7
  ],
  "decision_threshold": 0.5,
  "val_whole_dice_soft_macro": 0.1513317962056714,
  "val_whole_dice_hard": 0.2668513456046168,
  "val_whole_dice_hard_best_thr": 0.625,
  "val_whole_dice_hard_best_thr_score": 0.3244463

In [ ]:
# Export one manual probability map and segmentation from the fine-tuned ensemble.
trained_axes = [axis for axis, cfg in axis_cfgs.items() if cfg.checkpoint_path.exists()]
if not trained_axes:
    raise FileNotFoundError("No fine-tuned checkpoints exist yet. Run the training cell first.")

models = [seg.load_slice_block_model(axis_cfgs[axis], axis_cfgs[axis].checkpoint_path) for axis in trained_axes]
trained_cfgs = [axis_cfgs[axis] for axis in trained_axes]
case = mixed_val_atlas_cases[0] if mixed_val_atlas_cases else mixed_val_cases[0]
prob, mask, ref_img = seg.predict_ensemble_probability_map(models, trained_cfgs, case)
out_dir = RUN_DIR / "manual_atlas_ensemble_prediction"
seg.save_case_outputs(out_dir, case, prob, COMMON_CFG["DECISION_THRESHOLD"], ref_img)

print(f"Saved probability map and threshold segmentation to: {out_dir}")
print(f"Case: {case.case_id}")
print(f"Probability range: min={float(prob.min()):.6f}, max={float(prob.max()):.6f}")
print(f"Threshold voxels @ {COMMON_CFG['DECISION_THRESHOLD']:.2f}: {int((prob >= COMMON_CFG['DECISION_THRESHOLD']).sum())}")
seg.tf.keras.backend.clear_session()


Saved probability map and threshold segmentation to: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260430_141421_slice_blocks_resume_atlas_replay_looserfreeze/manual_atlas_ensemble_prediction
Case: ATLAS-Images-f0d7431e__ATLAS-Images-f0d7431e__sub-r011s017_ses-1_space-MNI152NLin2009aSym_T1w_MNI_norm
Probability range: min=0.000002, max=0.992528
Threshold voxels @ 0.50: 2615


: 